# SHIELD-ID: Data Collection
**Macro Shock Vulnerability Index — Indonesia (38 Provinsi)**

| # | Source | Data | Key Required |
|---|--------|------|--------------|
| 1 | World Bank Pink Sheet | Monthly commodity prices (oil, palm oil, nickel, coal, tin) | No |
| 2 | Yahoo Finance | Daily Brent/WTI prices; USD/IDR rate | No |
| 3 | FRED (St. Louis Fed) | Commodity indices, exchange rates | Yes (free) |
| 4 | BPS WebAPI (`webapi.bps.go.id`) | Inflasi, PDRB, Ekspor, Kemiskinan, Gini, Ketenagakerjaan — all 38 provinces | Yes (free) |

**BPS key:** register free at https://webapi.bps.go.id/developer/register → set `BPS_API_KEY` in `config.py`

In [ ]:
import os, re, sys, time, warnings
import requests
import pandas as pd
from io import BytesIO
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import config

try:
    import yfinance as yf
    YFINANCE_AVAILABLE = True
except ImportError:
    YFINANCE_AVAILABLE = False
    print("[SKIP] yfinance not installed. Run: pip install yfinance")

try:
    from fredapi import Fred
    FREDAPI_AVAILABLE = True
except ImportError:
    FREDAPI_AVAILABLE = False

print(f"Date range : {config.DATE_START} → {config.DATE_END}")
print(f"BPS key    : {'set ✓' if config.BPS_API_KEY else 'NOT SET — BPS will be skipped'}")
print(f"FRED key   : {'set ✓' if config.FRED_API_KEY else 'NOT SET — FRED will be skipped'}")

In [ ]:
# Create output directories
BASE_DIR  = os.path.dirname(os.path.abspath('__file__'))
DATA_ROOT = os.path.join(BASE_DIR, 'data', 'raw')

DIRS = {
    'commodities':    os.path.join(DATA_ROOT, 'commodities'),
    'exchange_rates': os.path.join(DATA_ROOT, 'exchange_rates'),
    'bps':            os.path.join(DATA_ROOT, 'bps'),
}
for path in DIRS.values():
    os.makedirs(path, exist_ok=True)
print('Directories ready:', list(DIRS.values()))

---
## 1. World Bank Pink Sheet
Monthly commodity price data going back to 1960, covering oil, metals, and agricultural commodities.
No API key required — direct Excel download.

In [ ]:
# Dynamically find current Pink Sheet URL (World Bank updates this link monthly)
def _get_pinksheet_url():
    page_url = "https://www.worldbank.org/en/research/commodity-markets"
    headers  = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        r = requests.get(page_url, timeout=30, headers=headers)
        r.raise_for_status()
        pattern = r'https://thedocs\.worldbank\.org[^\s"\'<>]*CMO-Historical-Data-Monthly\.xlsx'
        match = re.search(pattern, r.text)
        if match:
            print(f"  Found URL: {match.group(0)}")
            return match.group(0)
    except Exception as e:
        print(f"  [WARN] URL discovery failed ({e}) — falling back to config.py URL")
    return config.WORLDBANK_PINKSHEET_URL

print("Downloading World Bank Pink Sheet...")
url = _get_pinksheet_url()
try:
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    df_raw = pd.read_excel(BytesIO(r.content), sheet_name='Monthly Prices', header=4, index_col=0)

    col_map = {
        'Crude oil, avg, spot': 'oil_avg_usd_bbl',
        'Palm oil':             'palm_oil_usd_mt',
        'Nickel':               'nickel_usd_mt',
        'Coal, Australia':      'coal_australia_usd_mt',
        'Tin':                  'tin_usd_mt',
        'Copper':               'copper_usd_mt',
        'Natural gas, US':      'natgas_us_usd_mmbtu',
    }
    matched = {col: alias for col in df_raw.columns
               for key, alias in col_map.items() if key.lower() in str(col).lower()}

    df_wb = df_raw[list(matched.keys())].rename(columns=matched)
    df_wb.index = pd.to_datetime(df_wb.index, errors='coerce')
    df_wb = df_wb[df_wb.index.notna()].sort_index()
    df_wb = df_wb.loc[config.DATE_START:config.DATE_END]
    df_wb.index.name = 'date'

    out = os.path.join(DIRS['commodities'], 'worldbank_pinksheet.csv')
    df_wb.to_csv(out)
    print(f"Saved {len(df_wb)} rows, {len(df_wb.columns)} columns → {out}")
except Exception as e:
    print(f"[FAIL] {e}")
    df_wb = pd.DataFrame()

In [ ]:
# Preview
if not df_wb.empty:
    print(f"Shape: {df_wb.shape}")
    df_wb.tail()

---
## 2. Yahoo Finance
Daily prices for Brent crude (`BZ=F`), WTI crude (`CL=F`), and the USD/IDR spot rate (`USDIDR=X`).
No API key required.

In [ ]:
yahoo_dfs = {}
if not YFINANCE_AVAILABLE:
    print('[SKIP] yfinance not installed')
else:
    tickers = {
        'BZ=F':     ('commodities',    'yahoo_brent.csv',  'Brent Crude'),
        'CL=F':     ('commodities',    'yahoo_wti.csv',    'WTI Crude'),
        'USDIDR=X': ('exchange_rates', 'yahoo_usdidr.csv', 'USD/IDR'),
    }
    for ticker, (subdir, filename, label) in tickers.items():
        print(f"Downloading {label} ({ticker})...")
        df = yf.download(ticker, start=config.DATE_START, end=config.DATE_END,
                         progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.index.name = 'date'
        if not df.empty:
            out = os.path.join(DIRS[subdir], filename)
            df.to_csv(out)
            print(f"  Saved {len(df)} rows → {out}")
            yahoo_dfs[ticker] = df
        else:
            print(f"  [SKIP] Empty response for {ticker}")

In [ ]:
# Preview: Brent crude
if 'BZ=F' in yahoo_dfs:
    print(f"Brent crude shape: {yahoo_dfs['BZ=F'].shape}")
    yahoo_dfs['BZ=F'][['Close']].tail()

In [ ]:
# Preview: USD/IDR exchange rate
if 'USDIDR=X' in yahoo_dfs:
    print(f"USD/IDR shape: {yahoo_dfs['USDIDR=X'].shape}")
    yahoo_dfs['USDIDR=X'][['Close']].tail()

---
## 3. FRED API
St. Louis Federal Reserve economic data — commodity prices and exchange rates.

**Register free at:** https://fred.stlouisfed.org/docs/api/api_key.html  
**Then set `FRED_API_KEY` in `config.py`**

In [ ]:
FRED_SERIES = {
    'DCOILBRENTEU': 'brent_crude_daily_usd_bbl',
    'DCOILWTICO':   'wti_crude_daily_usd_bbl',
    'PNICKUSDM':    'nickel_monthly_usd_mt',
    'PCOALAUUSDM':  'coal_australia_monthly_usd_mt',
    'PPALMUSDM':    'palm_oil_monthly_usd_mt',
    'PTINUTUSDM':   'tin_monthly_usd_mt',
    'DEXINUS':      'usd_idr_daily',
}

df_fred = pd.DataFrame()

if not config.FRED_API_KEY:
    print('[SKIP] FRED_API_KEY not set in config.py')
    print('       Register free at: https://fred.stlouisfed.org/docs/api/api_key.html')
elif not FREDAPI_AVAILABLE:
    print('[SKIP] fredapi not installed. Run: pip install fredapi')
else:
    fred = Fred(api_key=config.FRED_API_KEY)
    frames = []
    for series_id, col_name in tqdm(FRED_SERIES.items(), desc='FRED series'):
        try:
            s = fred.get_series(series_id,
                                observation_start=config.DATE_START,
                                observation_end=config.DATE_END)
            s.name = col_name
            frames.append(s)
        except Exception as e:
            print(f'  [FAIL] {series_id}: {e}')

    if frames:
        df_fred = pd.concat(frames, axis=1)
        df_fred.index.name = 'date'
        out = os.path.join(DIRS['commodities'], 'fred_series.csv')
        df_fred.to_csv(out)
        print(f'Saved {len(df_fred)} rows, {len(df_fred.columns)} series → {out}')

In [ ]:
# Preview FRED data
if not df_fred.empty:
    print(f"FRED shape: {df_fred.shape}")
    df_fred.tail()

---
## 4. BPS WebAPI
Indonesian statistics via `webapi.bps.go.id` — the same JSON endpoint the BPS website calls internally.

**How var IDs were found:** each BPS statistics-table page has a "JSON" button that reveals:
```
https://webapi.bps.go.id/v1/api/list/model/data/lang/ind/domain/0000/var/{var_id}/th/{th}/key/{key}
```
- `domain/0000` = national scope (all 38 provinces returned as `vervar` rows)
- `var` = table ID (decoded from the base64 slug in the page URL)
- `th` = year code: `year − 1900` (e.g. 2024 → 124); max 2 years per call
- Response has `vervar` (provinces), `tahun` (years), `turtahun` (periods), `datacontent` (values keyed by concatenated codes)

**Free key:** https://webapi.bps.go.id/developer/register → set `BPS_API_KEY` in `config.py`

In [ ]:
BPS_WEBAPI_BASE = "https://webapi.bps.go.id/v1/api/list/model/data/lang/ind"

# var IDs decoded from page URL base64 slugs (e.g. MjI2MiMy → "2262#2" → var=2262)
# start_year: earliest year these specific table vars have data
BPS_TABLES = [
    {"key": "inflation_mom",       "var": 2262, "start_year": 2022,
     "output": "bps_inflation_mom.csv",
     "desc":   "Inflasi Bulanan M-to-M (2022=100), 38 Provinsi"},
    {"key": "inflation_ytd",       "var": 2387, "start_year": 2022,
     "output": "bps_inflation_ytd.csv",
     "desc":   "Inflasi Tahun Kalender Y-to-D (2022=100), 38 Provinsi"},
    {"key": "pdrb_adhb_quarterly", "var": 2268, "start_year": 2022,
     "output": "bps_pdrb_adhb_quarterly.csv",
     "desc":   "PDRB Triwulanan ADHB (seri 2010), Seluruh Provinsi (miliar Rp)"},
    {"key": "export_nonmigas",     "var": 2346, "start_year": 2022,
     "output": "bps_export_nonmigas.csv",
     "desc":   "Ekspor Non-Migas Bulanan per Provinsi (ribu USD)"},
    {"key": "export_migas",        "var": 2347, "start_year": 2022,
     "output": "bps_export_migas.csv",
     "desc":   "Ekspor Migas Bulanan per Provinsi (ribu USD)"},
    {"key": "poverty_p2",          "var": 504,  "start_year": 2005,
     "output": "bps_poverty_severity_p2.csv",
     "desc":   "Indeks Keparahan Kemiskinan P2 per Provinsi (%)"},
    {"key": "poverty_p1",          "var": 503,  "start_year": 2005,
     "output": "bps_poverty_depth_p1.csv",
     "desc":   "Indeks Kedalaman Kemiskinan P1 per Provinsi (%)"},
    {"key": "gini",                "var": 98,   "start_year": 2005,
     "output": "bps_gini.csv",
     "desc":   "Gini Ratio per Provinsi"},
    {"key": "employment_status",   "var": 2335, "start_year": 2022,
     "output": "bps_employment_by_status.csv",
     "desc":   "Penduduk Bekerja per Status Pekerjaan, per Provinsi (orang)"},
    {"key": "employment_sector",   "var": 2333, "start_year": 2022,
     "output": "bps_employment_by_sector.csv",
     "desc":   "Penduduk Bekerja per Lapangan Pekerjaan, per Provinsi (orang)"},
]


def _bps_th_chunks(start_year, end_year, chunk=2):
    """Yield BPS th-range strings (max 2 years per call = API limit)."""
    for yr in range(start_year, end_year + 1, chunk):
        yr_end = min(yr + chunk - 1, end_year)
        yield str(yr - 1900) if yr == yr_end else f"{yr - 1900}:{yr_end - 1900}"


def _bps_fetch_chunk(var_id, th_str, api_key):
    url = f"{BPS_WEBAPI_BASE}/domain/0000/var/{var_id}/th/{th_str}/key/{api_key}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        resp = r.json()
        if not isinstance(resp, dict):
            return None, "non-dict response"
        if resp.get("status", "").upper() != "OK":
            return None, f"API: {resp.get('message', resp.get('status', ''))}"
        return resp, None
    except Exception as e:
        return None, str(e)


def _bps_resp_to_df(resp, var_id):
    """Convert BPS response dict to long-format DataFrame.
    Key format: str(prov_val) + str(var_id) + str(turvar_val) + str(tahun_val) + str(turtahun_val)
    """
    vervar   = resp.get("vervar",   [])
    turvar   = resp.get("turvar",   [])
    tahun    = resp.get("tahun",    [])
    turtahun = resp.get("turtahun", [])
    content  = resp.get("datacontent", {})
    rows = []
    for prov in vervar:
        for tv in turvar:
            for th in tahun:
                for tth in turtahun:
                    key = (str(prov["val"]) + str(var_id) +
                           str(tv["val"]) + str(th["val"]) + str(tth["val"]))
                    val = content.get(key)
                    if val is not None:
                        rows.append({
                            "province_code": prov["val"],
                            "province":      prov["label"],
                            "area_type":     tv["label"],
                            "year":          th["label"],
                            "period":        tth["label"],
                            "value":         val,
                        })
    return pd.DataFrame(rows)


print(f"{len(BPS_TABLES)} BPS tables configured.")
print(f"BPS API key: {'set ✓' if config.BPS_API_KEY else '⚠ NOT SET'}")

In [ ]:
bps_results = {}   # key → DataFrame

if not config.BPS_API_KEY:
    print('[SKIP] BPS_API_KEY not set in config.py')
    print('       Register free at: https://webapi.bps.go.id/developer/register')
else:
    end_year = min(int(config.BPS_DATE_END[:4]), int(config.DATE_END[:4]))

    for meta in BPS_TABLES:
        desc      = meta["desc"]
        var_id    = meta["var"]
        out_path  = os.path.join(DIRS["bps"], meta["output"])
        start_yr  = meta["start_year"]

        print(f"\n[INFO] {desc} (var={var_id}, {start_yr}–{end_year})")
        frames = []

        for th_str in _bps_th_chunks(start_yr, end_year, chunk=2):
            resp, err = _bps_fetch_chunk(var_id, th_str, config.BPS_API_KEY)
            if err:
                print(f"  [FAIL] th={th_str}: {err}")
                continue
            chunk_df = _bps_resp_to_df(resp, var_id)
            if not chunk_df.empty:
                frames.append(chunk_df)
            time.sleep(0.4)

        if not frames:
            print(f"  [SKIP] No data returned")
            bps_results[meta["key"]] = pd.DataFrame()
            continue

        df = pd.concat(frames, ignore_index=True)
        df.to_csv(out_path, index=False)
        print(f"  [ OK ] {len(df):,} rows × {len(df.columns)} cols → {out_path}")
        bps_results[meta["key"]] = df

In [ ]:
# Preview: Inflation MoM (first BPS table)
if "inflation_mom" in bps_results and not bps_results["inflation_mom"].empty:
    df_preview = bps_results["inflation_mom"]
    print(f"Inflation MoM shape: {df_preview.shape}")
    print(f"Provinces: {df_preview['province'].nunique()}")
    print(f"Years: {sorted(df_preview['year'].unique())}")
    print(f"Periods: {sorted(df_preview['period'].unique())}")
    df_preview[df_preview['area_type'] == 'Tidak ada'].pivot_table(
        index='province', columns=['year', 'period'], values='value'
    ).head(10)
else:
    print('[NOTE] Inflation data not available — set BPS_API_KEY in config.py')

In [ ]:
print('\n=== COLLECTION SUMMARY ===')
summary = []
for subdir, path in DIRS.items():
    if os.path.isdir(path):
        for fname in sorted(os.listdir(path)):
            if fname.endswith('.csv'):
                fpath = os.path.join(path, fname)
                rows = sum(1 for _ in open(fpath, encoding='utf-8')) - 1
                summary.append({'folder': f'data/raw/{subdir}', 'file': fname, 'rows': rows})

if summary:
    display(pd.DataFrame(summary))
else:
    print('No files saved yet.')

if not config.BPS_API_KEY:
    print('\n[NOTE] BPS series skipped — set BPS_API_KEY in config.py')
    print('       Register: https://webapi.bps.go.id/developer/register')
if not config.FRED_API_KEY:
    print('\n[NOTE] FRED series skipped — set FRED_API_KEY in config.py')
    print('       Register: https://fred.stlouisfed.org/docs/api/api_key.html')